In [21]:
import json
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
from typing import List, Dict, Any
import warnings
warnings.filterwarnings('ignore')

from langchain.llms import OpenAI
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate, ChatPromptTemplate
from langchain.chains import LLMChain
from langchain.schema import BaseOutputParser
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

from dotenv import load_dotenv
load_dotenv()

False

In [22]:
import os
from langchain.chat_models import ChatOpenAI

os.environ["OPENAI_API_KEY"] = "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"  # Replace with your OpenRouter API key
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

llm = ChatOpenAI(
    model="mistralai/ministral-8b-2512",
    temperature=0.3,
    max_tokens=4000,
    openai_api_key=os.environ.get("OPENAI_API_KEY"),
    openai_api_base=os.environ.get("OPENAI_API_BASE")
)

print("✅ OpenRouter LLM configured successfully!")
print(f"🤖 Using model: {llm.model_name}")

✅ OpenRouter LLM configured successfully!
🤖 Using model: mistralai/ministral-8b-2512


In [23]:
def load_pr_data(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        return data
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return []

data_paths = {
    'jquery': r'PR data for Developers\jquery_pr_data_train.json',
}

all_pr_data = {}
for repo_name, file_path in data_paths.items():
    data = load_pr_data(file_path)
    all_pr_data[repo_name] = data

In [24]:
def analyze_developer_activity(all_pr_data):
    developer_stats = {}
    
    for repo_name, prs in all_pr_data.items():
        for pr in prs:
            author = pr.get('author', {}).get('username', 'unknown')
            if author not in developer_stats:
                developer_stats[author] = {
                    'total_prs': 0,
                    'repos': set(),
                    'pr_details': []
                }
            
            developer_stats[author]['total_prs'] += 1
            developer_stats[author]['repos'].add(repo_name)
            developer_stats[author]['pr_details'].append({
                'repo': repo_name,
                'title': pr.get('title', ''),
                'description': pr.get('description', ''),
                'labels': pr.get('labels', []),
                'state': pr.get('state', ''),
                'additions': pr.get('additions', 0),
                'deletions': pr.get('deletions', 0),
                'changed_files_count': pr.get('changed_files_count', 0),
                'language_stats': pr.get('language_stats', {}),
                'created_at': pr.get('created_at', '')
            })
    
    for author in developer_stats:
        developer_stats[author]['repos'] = list(developer_stats[author]['repos'])
    
    return developer_stats

developer_stats = analyze_developer_activity(all_pr_data)

In [25]:
class DeveloperProfile(BaseModel):
    developer_name: str = Field(description="Developer's username")
    experience_level: str = Field(description="Estimated experience level (Junior/Mid/Senior/Lead)")
    primary_skills: List[str] = Field(description="Top JavaScript-related skills inferred from PR evidence")
    programming_languages: List[str] = Field(description="Programming languages used (JavaScript-focused)")
    summary: str = Field(description="Overall JavaScript developer profile summary")
    identified_skills: Dict[str, List[str]] = Field(
        description="LLM-identified skill categories and specific skills (e.g., 'Backend Frameworks': ['Express.js', 'Node.js'])"
    )
    skill_evidence: List[str] = Field(
        description="Brief PR-based evidence snippets supporting the skill assessment"
    )

profile_parser = PydanticOutputParser(pydantic_object=DeveloperProfile)

skills_analysis_prompt = PromptTemplate(
    input_variables=["pr_data", "developer_name"],
    template="""Analyze the following GitHub Pull Request data for developer {developer_name} and extract their JavaScript technical skills and expertise.

PR Data:
{pr_data}

You are NOT given any predefined skill taxonomy. Infer skills directly from the PR evidence.

Look for evidence in PR titles, descriptions, labels, changed files, languages, and contribution patterns. Identify whatever JavaScript-related skills are actually demonstrated, such as:
- Frameworks, libraries, and runtimes (Express, React, Node.js, etc.)
- Language features and patterns (async/await, modules, prototypes, etc.)
- Backend, frontend, testing, DevOps, documentation, or tooling work
- Domain-specific expertise visible in the PRs

For each skill you identify, note which PR(s) or signals support it.

Provide a detailed, evidence-based analysis of this developer's JavaScript ecosystem skills.""",
    partial_variables={"format_instructions": "Provide a detailed JavaScript-focused technical analysis grounded in PR evidence."}
)

profile_generation_prompt = PromptTemplate(
    input_variables=["skills_analysis", "developer_name", "pr_count", "repo_list"],
    template="""Based on the following JavaScript skills analysis, generate a comprehensive JavaScript developer profile.

Developer: {developer_name}
Total PRs: {pr_count}
Repositories: {repo_list}

Skills Analysis:
{skills_analysis}

Generate a JavaScript developer profile that includes:
1. Experience level assessment (Junior/Mid/Senior/Lead) based on PR contributions
2. Primary JavaScript technical skills and frameworks demonstrated in the PRs
3. Programming languages used
4. Overall JavaScript developer summary
5. identified_skills: your own skill taxonomy inferred from the PR evidence
6. skill_evidence: short PR-based evidence snippets supporting the assessment

========== RULES FOR identified_skills ==========
- Create your own meaningful category names based on what the PRs actually show
- Do NOT use any predefined KU/Knowledge Unit taxonomy
- Include specific technologies, patterns, and practices you can support from the PR data
- Use empty arrays [] for categories with no supporting evidence
- Use plain skill names only — do NOT append counts, frequency, or numeric metadata
- Prefer concrete skills (e.g., "Express.js", "Middleware", "Dependency Upgrades") over vague labels

Example identified_skills:
{{
  "Backend Frameworks": ["Express.js", "Node.js"],
  "Dependency Management": ["npm", "semver upgrades"],
  "Documentation & Community": ["contributor guidelines", "triager role"],
  "Testing": ["Jest"],
  "Asynchronous Programming": ["Promises", "async/await"]
}}

========== RULES FOR skill_evidence ==========
- Provide 3-8 short bullet-style strings
- Each item should reference a concrete PR signal (title, label, description, or change type)
- Do not invent evidence that is not supported by the analysis

========== CRITICAL RULES FOR JSON OUTPUT ==========
1. Return ONLY raw valid JSON — no markdown code fences (no ```json)
2. Do NOT include comments (no // or /* */)
3. Use \\n for line breaks inside string values, not literal newlines
========== END JSON OUTPUT RULES ==========

{format_instructions}

Be specific and evidence-based. Only include skills and categories supported by the PR analysis.""",
    partial_variables={
        "format_instructions": profile_parser.get_format_instructions(),
    }
)

In [26]:
def create_analysis_chains(llm):
    
    skills_chain = LLMChain(
        llm=llm,
        prompt=skills_analysis_prompt,
        verbose=True
    )
    
    profile_chain = LLMChain(
        llm=llm,
        prompt=profile_generation_prompt,
        verbose=True
    )
    
    return skills_chain, profile_chain

def prepare_pr_data_for_analysis(developer_data, max_prs=20):
    """Prepare PR data for LLM analysis (limit size to avoid token limits)"""
    
    pr_details = developer_data['pr_details'][:max_prs] 
    
    pr_summary = []
    for pr in pr_details:
        summary = {
            'title': pr['title'][:100],
            'repo': pr['repo'],
            'labels': pr['labels'],
            'state': pr['state'],
            'additions': pr['additions'],
            'deletions': pr['deletions'],
            'files_changed': pr['changed_files_count'],
            'languages': list(pr['language_stats'].keys()) if pr['language_stats'] else []
        }
        
        if pr['description'] and len(pr['description']) < 500:
            summary['description'] = pr['description'][:300]
        
        pr_summary.append(summary)
    
    return pr_summary

if llm and developer_stats:
    print(f"🔧 Creating analysis chains for ALL developers...")
    
    skills_chain, profile_chain = create_analysis_chains(llm)
    
    eligible_developers = [
        (name, stats) for name, stats in developer_stats.items()
        if stats['total_prs'] >= 3 and not name.endswith('[bot]') and 'bot' not in name.lower()
    ]
    
    eligible_developers.sort(key=lambda x: x[1]['total_prs'], reverse=True)
    
    print(f"✅ Chains created successfully")
    print(f"👥 Found {len(eligible_developers)} eligible developers")
    print(f"📊 Will process developers with 3+ PRs (excluding bots)")
    
elif llm:
    print("⚠️  LLM configured but no developers found")
    print("Please check that PR data was loaded correctly")
else:
    print("⚠️  LLM not configured - please run the LLM configuration cell first")
    print("Set up your Gemini API key to continue")

🔧 Creating analysis chains for ALL developers...
✅ Chains created successfully
👥 Found 97 eligible developers
📊 Will process developers with 3+ PRs (excluding bots)


In [27]:
def clean_llm_json_response(text: str) -> str:
    text = text.strip()
    fence_match = re.search(r'```(?:json)?\s*(.*?)\s*```', text, re.DOTALL | re.IGNORECASE)
    if fence_match:
        text = fence_match.group(1).strip()
    elif text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*', '', text)
        text = re.sub(r'\s*```$', '', text).strip()

    cleaned_lines = []
    for line in text.split('\n'):
        cleaned_lines.append(re.sub(r'//.*$', '', line))
    text = '\n'.join(cleaned_lines)
    text = re.sub(r',\s*([}\]])', r'\1', text)
    return text


def repair_multiline_json_strings(text: str) -> str:
    result = []
    in_string = False
    escape = False

    for char in text:
        if in_string:
            if escape:
                result.append(char)
                escape = False
            elif char == '\\':
                result.append(char)
                escape = True
            elif char == '"':
                in_string = False
                result.append(char)
            elif char in '\n\r':
                result.append('\\n')
            else:
                result.append(char)
        else:
            if char == '"':
                in_string = True
            result.append(char)

    return ''.join(result)


def normalize_profile_data(data: dict) -> dict:
    summary = data.get("summary")
    if isinstance(summary, dict):
        data["summary"] = (
            summary.get("summary")
            or summary.get("text")
            or json.dumps(summary)
        )
    return data


def parse_profile_response(profile_result: str) -> DeveloperProfile:
    cleaned = clean_llm_json_response(profile_result)
    candidates = [cleaned, repair_multiline_json_strings(cleaned)]

    last_error = None
    for candidate in candidates:
        try:
            return profile_parser.parse(candidate)
        except Exception as parse_error:
            last_error = parse_error
            try:
                data = normalize_profile_data(json.loads(candidate))
                return DeveloperProfile(**data)
            except Exception as json_error:
                last_error = json_error

    raise last_error


def generate_developer_profile(developer_name, developer_data, pr_summary):
    try:
        skills_analysis_input = {
            "developer_name": developer_name,
            "pr_data": json.dumps(pr_summary, indent=2)
        }
        
        print(f"  🔍 Analyzing skills for {developer_name}...")
        skills_analysis_result = skills_chain.run(skills_analysis_input)
        
        profile_input = {
            "developer_name": developer_name,
            "skills_analysis": skills_analysis_result,
            "pr_count": developer_data['total_prs'],
            "repo_list": ', '.join(developer_data['repos'])
        }
        
        print(f"  🤖 Generating profile for {developer_name}...")
        profile_result = profile_chain.run(profile_input)
        
        try:
            parsed_profile = parse_profile_response(profile_result)
            profile_dict = parsed_profile.dict()
        except Exception as parse_error:
            print(f"  ⚠️  Profile parsing failed for {developer_name}: {parse_error}")
            profile_dict = {
                "developer_name": developer_name,
                "raw_response": profile_result,
                "parsing_error": str(parse_error)
            }
        
        return {
            "success": True,
            "profile": profile_dict,
            "skills_analysis": skills_analysis_result,
            "raw_profile": profile_result
        }
        
    except Exception as e:
        print(f"  ❌ Error generating profile for {developer_name}: {e}")
        return {
            "success": False,
            "error": str(e),
            "developer": developer_name
        }

In [28]:
import time

def load_checkpoint(checkpoint_file="profile_generation_checkpoint.json"):
    """Load the last checkpoint to resume processing"""
    if os.path.exists(checkpoint_file):
        try:
            with open(checkpoint_file, 'r', encoding='utf-8') as f:
                checkpoint = json.load(f)
            return checkpoint
        except Exception as e:
            print(f"⚠️  Error loading checkpoint: {e}")
            return None
    return None

def save_checkpoint(processed_developers, checkpoint_file="profile_generation_checkpoint.json"):
    """Save current progress checkpoint"""
    checkpoint = {
        "processed_developers": processed_developers,
        "last_updated": datetime.now().isoformat(),
        "total_processed": len(processed_developers)
    }
    
    try:
        with open(checkpoint_file, 'w', encoding='utf-8') as f:
            json.dump(checkpoint, f, indent=2)
        print(f"💾 Checkpoint saved: {len(processed_developers)} developers processed")
    except Exception as e:
        print(f"❌ Error saving checkpoint: {e}")

def get_remaining_developers(eligible_developers, processed_developers=None):
    """Get list of developers that haven't been processed yet"""
    if not processed_developers:
        return eligible_developers
    
    processed_names = set(processed_developers)
    remaining = [(name, stats) for name, stats in eligible_developers 
                 if name not in processed_names]
    
    return remaining

def batch_generate_profiles_with_resume(eligible_developers, batch_size=15, delay_seconds=2, 
                                      checkpoint_file="profile_generation_checkpoint.json"):
    """Generate profiles with checkpoint/resume capability"""
    
    # Load existing checkpoint
    checkpoint = load_checkpoint(checkpoint_file)
    processed_developers = checkpoint.get('processed_developers', []) if checkpoint else []
    
    # Get remaining developers to process
    remaining_developers = get_remaining_developers(eligible_developers, processed_developers)
    
    if not remaining_developers:
        print("🎉 All developers have already been processed!")
        return []
    
    print(f"📊 RESUME STATUS:")
    print(f"   ✅ Already processed: {len(processed_developers)} developers")
    print(f"   ⏳ Remaining to process: {len(remaining_developers)} developers")
    print(f"   🎯 This batch will process: {min(batch_size, len(remaining_developers))} developers")
    
    # Process only the batch size
    current_batch = remaining_developers[:batch_size]
    batch_results = []
    successful_profiles = 0
    failed_profiles = 0
    
    print(f"\n🚀 Starting batch processing ({len(current_batch)} developers)...")
    
    for i, (dev_name, dev_data) in enumerate(current_batch, 1):
        try:
            print(f"\nProcessing {i}/{len(current_batch)}: {dev_name} ({dev_data['total_prs']} PRs)")
            
            pr_summary = prepare_pr_data_for_analysis(dev_data, max_prs=15)
            result = generate_developer_profile(dev_name, dev_data, pr_summary)
            
            if result["success"]:
                batch_results.append({
                    "developer": dev_name,
                    "result": result,
                    "stats": dev_data,
                    "processed_at": datetime.now().isoformat()
                })
                
                # Add to processed list and save checkpoint immediately
                processed_developers.append(dev_name)
                save_checkpoint(processed_developers, checkpoint_file)
                
                successful_profiles += 1
                print(f"  ✅ Success - Profile generated and checkpoint saved")
            else:
                failed_profiles += 1
                print(f"  ❌ Failed: {result.get('error', 'Unknown')}")
                
        except Exception as e:
            failed_profiles += 1
            print(f"  💥 Exception: {e}")
        
        # Delay between API calls (except for the last one)
        if i < len(current_batch):
            print(f"  ⏸️  Waiting {delay_seconds} seconds...")
            time.sleep(delay_seconds)
    
    # Final summary
    total_processed_now = len(processed_developers)
    total_remaining = len(remaining_developers) - len(current_batch)
    
    print(f"\n🎉 BATCH COMPLETED!")
    print(f"✅ This batch successful: {successful_profiles}")
    print(f"❌ This batch failed: {failed_profiles}")
    print(f"📊 Total processed so far: {total_processed_now}/{len(eligible_developers)} developers")
    print(f"⏳ Still remaining: {total_remaining} developers")
    
    if total_remaining > 0:
        print(f"\n💡 To continue processing, run the next cell again!")
        print(f"   Next batch will process up to {min(batch_size, total_remaining)} more developers")
    else:
        print(f"\n🎊 ALL DEVELOPERS COMPLETED! Cleaning up checkpoint...")
        # Optionally remove checkpoint file when done
        if os.path.exists(checkpoint_file):
            os.remove(checkpoint_file)
    
    return batch_results

def export_profiles_from_checkpoint(checkpoint_file="profile_generation_checkpoint.json", 
                                  output_dir="generated_profiles"):
    """Export all profiles that have been generated so far"""
    
    # Load all existing profile files
    if not os.path.exists(output_dir):
        print(f"📁 Output directory {output_dir} doesn't exist yet")
        return []
    
    existing_files = [f for f in os.listdir(output_dir) if f.endswith('_profile.json')]
    print(f"💾 Found {len(existing_files)} existing profile files in {output_dir}/")
    
    return existing_files

def export_batch_profiles(batch_results, output_dir="Developer's Profiles"):
    """Export profiles from a single batch"""
    if not batch_results:
        return []
    
    os.makedirs(output_dir, exist_ok=True)
    exported_files = []
    
    print(f"💾 Exporting {len(batch_results)} new profiles...")
    
    for i, profile_data in enumerate(batch_results, 1):
        dev_name = profile_data["developer"]
        result = profile_data["result"]
        stats = profile_data["stats"]
        
        export_data = {
            "metadata": {
                "developer_name": dev_name,
                "generation_timestamp": profile_data["processed_at"],
                "llm_provider": f"OpenRouter ({llm.model_name})",
                "source": "GitHub PR Data Analysis (LLM-inferred skills, no KU taxonomy)",
                "total_prs_analyzed": stats['total_prs'],
                "repositories": stats['repos'],
                "pr_data_sample_size": min(15, len(stats['pr_details']))
            },
            "profile": result.get("profile"),
            "statistics": {
                "total_prs": stats['total_prs'],
                "repos_contributed": len(stats['repos']),
                "repo_list": stats['repos']
            }
        }
        
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"{dev_name}_profile_{timestamp}.json"
        filepath = os.path.join(output_dir, filename)
        
        try:
            with open(filepath, 'w', encoding='utf-8') as f:
                json.dump(export_data, f, indent=2, ensure_ascii=False)
            
            exported_files.append(filepath)
            
        except Exception as e:
            print(f"❌ Export failed for {dev_name}: {e}")
    
    print(f"💾 Exported {len(exported_files)} profile files to {output_dir}/")
    return exported_files


def regenerate_single_profile(developer_name, output_dir="Developer's Profiles"):
    """Regenerate and export a profile for one developer."""
    if not (llm and developer_stats and 'skills_chain' in globals() and 'profile_chain' in globals()):
        raise RuntimeError("Run setup cells first (LLM, data load, chains)")

    if developer_name not in developer_stats:
        raise ValueError(f"{developer_name} not found in developer_stats")

    dev_data = developer_stats[developer_name]
    pr_summary = prepare_pr_data_for_analysis(dev_data)

    print(f"🔄 Regenerating profile for {developer_name}...")
    result = generate_developer_profile(developer_name, dev_data, pr_summary)

    profile = result.get("profile", {})
    if profile.get("parsing_error"):
        print(f"⚠️  Parsing issue: {profile['parsing_error']}")
    else:
        print(f"✅ Profile parsed successfully ({profile.get('experience_level', 'Unknown')} level)")

    batch_results = [{
        "developer": developer_name,
        "result": result,
        "stats": dev_data,
        "processed_at": datetime.now().isoformat()
    }]

    exported_files = export_batch_profiles(batch_results, output_dir=output_dir)
    print(f"💾 Saved: {exported_files[0] if exported_files else 'export failed'}")
    return result, exported_files

In [29]:
def check_progress_status():
    """Check current progress status"""
    checkpoint = load_checkpoint()
    
    if not checkpoint:
        print("📊 No checkpoint found - starting fresh")
        print(f"🎯 Total developers to process: {len(eligible_developers)}")
        return
    
    processed = len(checkpoint['processed_developers'])
    total = len(eligible_developers)
    remaining = total - processed
    percentage = (processed / total) * 100
    
    print("📊 PROGRESS STATUS")
    print("=" * 40)
    print(f"✅ Completed: {processed}/{total} developers ({percentage:.1f}%)")
    print(f"⏳ Remaining: {remaining} developers")
    print(f"📅 Last updated: {checkpoint['last_updated']}")
    
    if remaining > 0:
        print(f"\n💡 Run the next cell to continue processing")
    else:
        print(f"🎉 All developers completed!")

# Enhanced usage function
def run_profile_generation_session(batch_size=15, delay_seconds=2):
    """Run a single profile generation session"""
    
    if not (llm and developer_stats and 'eligible_developers' in globals()):
        print("❌ Please run the setup cells first (LLM configuration and developer analysis)")
        return
    
    print("🎯 STARTING PROFILE GENERATION SESSION")
    print("=" * 60)
    
    batch_results = batch_generate_profiles_with_resume(
        eligible_developers=eligible_developers,
        batch_size=batch_size,
        delay_seconds=delay_seconds
    )
    
    # Export this batch if we got results
    if batch_results:
        exported_files = export_batch_profiles(batch_results)
        print(f"💾 Exported {len(exported_files)} new profile files")
    
    # Show status of all profiles
    existing_profiles = export_profiles_from_checkpoint()
    
    return batch_results

# Check current status
print("🔍 CHECKING CURRENT PROGRESS...")
check_progress_status()

🔍 CHECKING CURRENT PROGRESS...
📊 No checkpoint found - starting fresh
🎯 Total developers to process: 97


# Run Profile Generation in Batches

In [31]:

BATCH_SIZE = 60     
DELAY_SECONDS = 2      


batch_results = run_profile_generation_session(
    batch_size=BATCH_SIZE, 
    delay_seconds=DELAY_SECONDS
)


if batch_results:
    print(f"\n📋 Quick Preview of This Batch:")
    for i, profile_data in enumerate(batch_results[:3], 1):
        dev_name = profile_data["developer"]
        profile = profile_data["result"]["profile"]
        if profile:
            experience = profile.get('experience_level', 'Unknown')
            skills_count = len(profile.get('primary_skills', []))
            category_count = len(profile.get('identified_skills', {}))
            evidence_count = len(profile.get('skill_evidence', []))
            print(f"  {i}. {dev_name}: {experience} level, {skills_count} primary skills, {category_count} LLM categories, {evidence_count} evidence items")
    
    if len(batch_results) > 3:
        print(f"  ... and {len(batch_results) - 3} more profiles generated")
else:
    print("📝 No new profiles generated in this batch")

🎯 STARTING PROFILE GENERATION SESSION
📊 RESUME STATUS:
   ✅ Already processed: 40 developers
   ⏳ Remaining to process: 57 developers
   🎯 This batch will process: 57 developers

🚀 Starting batch processing (57 developers)...

Processing 1/57: mixed (5 PRs)
  🔍 Analyzing skills for mixed...


> Entering new LLMChain chain...
Prompt after formatting:
Analyze the following GitHub Pull Request data for developer mixed and extract their JavaScript technical skills and expertise.

PR Data:
[
  {
    "title": "Core:Suggest remove unnecessary parameter",
    "repo": "jquery",
    "labels": [],
    "state": "closed",
    "additions": 0,
    "deletions": 0,
    "files_changed": 0,
    "languages": [],
    "description": "I removed unnecessary parameter for constructor of \"end\" method. It`s because jQuery constructor with no parameter is the same as jQuery constructor with null parameter.\n\n<!-- Reviewable:start -->\n\n[<img src=\"https://reviewable.io/review_button.png\" height=40 alt=\"Revi